# M16 — Verified Baselines & Cross-Domain Validation (Colab)

**Status: tasks T04, T06, T09, T10 authored (M1 verification, NFL target family + reliability + freeze, baseline ladder, detectability, confirmatory retrieval, leakage audit, G16) — never executed by the author.**

## A. Introduction

- **Question.** Is tracking-derived information reliable and incremental beyond position and official Combine testing, on held-out athletes — and does cross-domain similarity survive position and context controls?
- **Workstream A.** (1) Verify the official **M1** (`draft_year, height, weight, forty, ten`) on **`nfl.q90_shared`** at 1e-8 against its own refit and every compatible persisted official OOF record; (2) build NFL task targets T1–T6 from NFL data only, measure their reliability / exposure / context dependence and freeze the eligible ones once; (3) baseline ladder MEAN · POS · M1 · M1+F0 · M1+simple · M1+M14_reliable (+ ENet / trees) on one cohort and fold system; (4) minimum detectable effect.
- **Workstream B.** Five-fold retrieval of the M15-gate-eligible representations (pools all / same position / twins), F0 and random controls, position-only expectation, CEM sensitivity, leakage audit.
- **Two targets, never mixed.** `nfl.q90_shared` (T0) and `nfl_speed_q90_median` (T0x) are audited separately; their RMSEs are never compared.
- **Inputs.** Bundle; M12/M13/M14 experiments (persisted OOF), M14 run (athlete metrics + measurement status), M15 run (sequence features, embeddings, gate) — all read only.
- **Gate.** G16 A–D computed from the saved tables; NOT_EVALUATED wherever evidence is missing.

## B. Setup

In [ ]:
# Setup 1 — mount Drive and install nflbowl from GitHub (the Colab PyTorch / CUDA build is never replaced)
from google.colab import drive
drive.mount('/content/drive')
REPO = 'git+https://github.com/mybenkhadda/NFL_bowl.git@main'
%pip install -q {REPO}
%pip install -q --upgrade --force-reinstall --no-deps {REPO}
%pip install -q markdown

In [ ]:
# Setup 2 — version check (after a reinstall: Runtime → Restart session, then continue from here)
import nflbowl
print('nflbowl', nflbowl.__version__)
assert tuple(int(x) for x in nflbowl.__version__.split('.')[:2]) >= (0, 7), 'nflbowl < 0.7.0 is imported: restart the session and rerun from Setup 2'
import json
import polars as pl
from IPython.display import Markdown, display
from nflbowl.lab import handoff
from nflbowl.lab import status as S
from nflbowl.lab.m16 import config as m16cfg, m1_verification as m1v, plots as m16plots, targets as m16targets

In [ ]:
# Setup 3 — configuration (the ONLY cell to edit). M12 / M13 / M14 / M15 directories are read only.
CFG = {
    'experiment_name': 'm16_verified_baselines_v1',
    'bundle_dir': '/content/drive/MyDrive/nfl-big-data-bowl-2027/bundles/m12f_v1_clean',
    'drive_root': '/content/drive/MyDrive/NFLBowl',
    'extra_oof_sources': [],               # further experiment folders holding evaluation/records/transfer_oof__*.parquet
    'exploratory_target_path': '',         # parquet / csv with nfl_id + nfl_speed_q90_median (exploratory notebook export)
    'reproduction_tol': 1e-8,
    'seed': 42,
}
cfg = m16cfg.resolve(CFG)
run = m16cfg.M16Run(cfg.run_dir, cfg)
run.snapshot_config(cfg, handoff.environment(cfg.m13()))
print('run dir                 ', run.root)
print('config fingerprint      ', cfg.fingerprint())
print('persisted-OOF sources   ', [str(p) for p in cfg.experiment_dirs()] or 'none found (G16-B persisted criterion will be NOT_EVALUATED)')
display(run.ledger.table())

In [ ]:
# Setup 4 — the M16 experiment (new directory under RUN_DIR/checkpoints; refuses another bundle or science hash)
def step_open(ctx):
    return handoff.open_experiment(cfg.m13(), run)

EXP = S.run_step(run, run.ledger, 'E16.open', step_open, note='open the M16 experiment on the official bundle / science')
if EXP is None:
    display(run.ledger.table())
    raise SystemExit('the experiment could not be opened — see the reason above and logs/')
print('experiment', EXP.id, '· bundle', EXP.bundle.fingerprint, '· science', EXP.cfg.science_hash())

## C. Data audit — E16.A0 M1 contract · E16.A1 reproducibility (T04)

One guarded step: contract audit → official ladder recomputation (`Experiment.transfer_predictions(models=[])`) → independent refit (`transfer_fairness.baseline_predictions`) → persisted-record discovery and comparison → fold / target checks → tables, audit report, G16-B rows. On a mismatch the step ends FAILED_VALIDATION **after** writing all evidence.

In [ ]:
def step_m1(ctx):
    v = m1v.verify_m1(cfg, run, EXP, ctx)
    ctx['outputs'] += ['tables/baselines/m1_reproducibility', 'manifests/m1_verification.json', 'reports/markdown/m1_audit_report.md']
    return v

M1 = S.run_step(run, run.ledger, 'E16.A1', step_m1, note='official M1 contract + reproducibility', requires=['E16.open'])
summary_path = run.root / 'manifests' / 'm1_verification.json'
SUMMARY = json.loads(summary_path.read_text()) if summary_path.is_file() else None
display(run.ledger.table().filter(pl.col('step') == 'E16.A1'))
if SUMMARY:
    print('VERDICT:', SUMMARY['verdict'], '| internal ok:', SUMMARY['internal_ok'], '| checks ok:', SUMMARY['checks_ok'],
          '| compatible persisted records:', SUMMARY['n_compatible_records'], '| persisted ok:', SUMMARY['persisted_ok'])
    print('failures:', SUMMARY['failures'] or 'none')

In [ ]:
# Evidence tables (re-read from disk, so they also show after a FAILED_VALIDATION)
for name in ('m1_contract_columns', 'm1_reproducibility', 'm1_checks', 'm1_persisted_records', 'm1_metrics'):
    if run.has(name, 'baselines'):
        display(Markdown(f'**{name}**')); display(run.frame(name, 'baselines'))
if run.has('m1_athlete_diffs', 'baselines'):
    d = run.frame('m1_athlete_diffs', 'baselines')
    display(Markdown('**mismatched athletes**')); display(d.filter(pl.col('mismatch')).head(50))

In [ ]:
# Figures (PNG / SVG / PDF + source data)
if run.has('m1_athlete_diffs', 'baselines'):
    fig, dat = m16plots.reproduction_diffs(run.frame('m1_athlete_diffs', 'baselines'), cfg.reproduction_tol)
    run.figure(fig, 'M16_A1_01_m1_reproduction_diffs', data=dat, question='Is the official M1 reproduced athlete by athlete?',
               caption='|Δ prediction| per athlete and comparison (log scale; exact zeros drawn at 1e-17); dashed line = tolerance.')
if run.has('m1_metrics', 'baselines'):
    fig, dat = m16plots.metrics_by_fold(run.frame('m1_metrics', 'baselines'))
    run.figure(fig, 'M16_A1_02_rmse_by_fold', data=dat, question='Do recomputed and persisted baselines agree fold by fold?',
               caption='Out-of-fold RMSE by fold on nfl.q90_shared (descriptive; equality is judged on predictions).')

## D. Exploratory target audit — T0x `nfl_speed_q90_median` (kept separate)

Coverage, distribution and fingerprint of the exploratory target next to the official one. No model is fitted on it here and no RMSE is compared. Without `exploratory_target_path` (and no such column in `athletes.parquet`) the step is **BLOCKED_MISSING_INPUT**.

In [ ]:
def step_t0x(ctx):
    return m1v.exploratory_target_audit(cfg, EXP.bundle.athletes, run)

T0X = S.run_step(run, run.ledger, 'E16.A0-x', step_t0x, note='exploratory target audit (descriptive)', requires=['E16.open'])
if T0X is not None:
    display(run.frame('target_registry_t0_t0x', 'target'))
    print({k: T0X[k] for k in ('overlap_athletes', 'only_T0', 'only_T0x', 'note')})
else:
    display(run.ledger.table().filter(pl.col('step') == 'E16.A0-x'))

In [ ]:
from nflbowl.lab.m16 import plots as m16plots

## E. Experiments — Workstream A: E16.A2–A3 NFL target family and its reliability (T06)

Targets from NFL data only (no Combine value read): **T1** high-speed capacity (q90 window max speed per demand kind, demand-mix weighted), **T2 / T2b** acceleration at matched entry speed (2–4 / 4–6 yd/s), **T3** braking (M14 `max_controlled_decel`, complete events), **T4** reacceleration (`first_step_explosiveness`), **T5** turn-speed retention, **T6** within-athlete consistency. Then split-half reliability (season → game → random, labelled), exposure curves and dependence, demand-kind share, **G16-A**, and the one-time freeze `manifests/target_freeze_m16.json`. T0 (`nfl.q90_shared`) remains the primary target; it is not rebuilt here.

In [ ]:
# Shared inputs for Workstreams A and B: M15 data view on the opened bundle + E15.A1 sequence features (M15 run, read only)
from nflbowl.lab.m15 import plots as m15plots
from nflbowl.lab.m16 import pipeline as m16pipe
CFG15 = m16pipe.m15_config(cfg)
DATA16 = m16pipe.data_from_experiment(EXP, CFG15)
FEAT = m16pipe.feature_table(cfg, DATA16, CFG15, run)
print('sequences', DATA16.index.height, '· feature rows', FEAT.height)

# E16.A2 — NFL target family T1–T6 (windows + M14 detector on NFL rows), coverage, redundancy
def step_a2(ctx):
    return m16pipe.run_targets(cfg, DATA16, FEAT, CFG15, run, ctx)

T = S.run_step(run, run.ledger, 'E16.A2', step_a2, note='NFL task targets', requires=['E16.open'])
if T is not None:
    display(run.frame('definitions', 'target').select('target_id', 'column', 'unit', 'role', 'aggregation'))
    display(run.frame('coverage', 'target').filter(pl.col('position_group') == 'ALL'))

In [ ]:
if T is not None:
    fig, dat = m16plots.target_coverage(run.frame('coverage', 'target'))
    run.figure(fig, 'M16_A2_01_target_coverage', data=dat, question='Which NFL targets are observable, for how many athletes?',
               caption='Athletes per target and position after the minimum-exposure rules.')

In [ ]:
# E16.A3 — reliability, exposure, context share, G16-A and the immutable freeze
def step_a3(ctx):
    if T is None:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'targets not available (run E16.A2)', missing=['E16.A2'])
    return m16pipe.run_reliability(cfg, DATA16, T, run, ctx)

REL = S.run_step(run, run.ledger, 'E16.A3', step_a3, note='target reliability + freeze', requires=['E16.A2'])
if REL is not None:
    display(REL['reliability'])
    display(REL['exposure'])
    display(REL['context'])
    display(Markdown('**G16-A per target**')); display(REL['gate_a'])
    print('freeze:', {k: REL['freeze'].get(k) for k in ('primary', 'eligible_task_targets', 'conflict', 'written_at')})

In [ ]:
if REL is not None:
    fig, dat = m16plots.reliability_vs_exposure(REL['curve'])
    run.figure(fig, 'M16_A3_01_reliability_vs_exposure', data=dat, question='How much NFL exposure makes a target reliable?',
               caption='Split-half reliability (Spearman–Brown, position-residualised) vs observations per half; dashed 0.7.')

## E16.A4–A6 — baseline ladder, within-position ranking and detectability

Official engine (`ae.oof_ladder`) for MEAN, POS, M1, M1+F0, M1+simple (M15 E15.A1 athlete medians, train-coverage factory) and M1+M14_reliable (M14 `validated` metrics); ENet (and LightGBM / CatBoost when installed) on identical rows. Strict cohort, paired fold-stratified bootstrap, Holm on the T0 primary family {M1 vs POS, M1+simple vs M1, M1+M14_reliable vs M1}, precise-null rule, within-position Spearman / concordance. Then the planted-signal MDE for every ladder target. Task targets enter only when frozen as eligible (G16-A), unless `task_targets_in_ladder='all'` (labelled).

In [ ]:
# E16.A4 / A5 — baseline ladder per target (T0 + frozen eligible task targets), registry, within-position ranking
def step_a4(ctx):
    if T is None:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'targets not available (run E16.A2)', missing=['E16.A2'])
    return m16pipe.run_baselines(cfg, EXP, FEAT, T['targets'], REL['freeze'] if REL else None, run, ctx)

BASE = S.run_step(run, run.ledger, 'E16.A4', step_a4, note='learner ladder MEAN · POS · M1 · M1+F0 · M1+simple · M1+M14_reliable (+ ENet / trees)',
                  requires=['E16.A2', 'E16.A1'])
if BASE is not None:
    display(run.frame('baseline_notes', 'baselines'))
    display(BASE['metrics'].select('target_id', 'model', 'n_athletes', 'rmse', 'r2', 'skill_vs_position_oof', 'spearman'))
    display(Markdown('**Declared status vs M1**')); display(BASE['status'])

In [ ]:
if BASE is not None and BASE['paired'].height:
    fig, dat = m16plots.ladder_forest(BASE['paired'], 'M1')
    run.figure(fig, 'M16_A4_01_ladder_forest', data=dat, question='Does anything beat the official M1 baseline?',
               caption='ΔRMSE vs M1 per target and model (> 0 = better), 95 % fold-stratified athlete bootstrap; grey = ±2 % margin.')
    if BASE['ranking'].height:
        for tid in BASE['ranking']['target_id'].unique().to_list():
            fig, dat = m16plots.within_position_ranking(BASE['ranking'].filter(pl.col('target_id') == tid))
            run.figure(fig, f'M16_A5_01_within_position_{tid}', data=dat, question='Does tracking improve ranking within a position?',
                       caption='Within-position Spearman with 95 % athlete-bootstrap CI.')

In [ ]:
# E16.A6 — minimum detectable effect on the real cohort structure (resumable per target)
def step_a6(ctx):
    if BASE is None:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'analysis frame not available (run E16.A4)', missing=['E16.A4'])
    return m16pipe.run_detectability(cfg, BASE['data'], m16pipe.ladder_targets(cfg, REL['freeze'] if REL else None), run, ctx)

MDE = S.run_step(run, run.ledger, 'E16.A6', step_a6, note='planted-signal power / MDE', requires=['E16.A4'])
if MDE is not None:
    display(MDE['mde'])

In [ ]:
if MDE is not None:
    fig, dat = m16plots.power_curves(MDE['power'])
    run.figure(fig, 'M16_A6_01_power_curves', data=dat, question='What effect could this cohort detect?',
               caption=f"Power vs planted variance share (feature reliability {cfg.detect_reliability}); dashed 0.8.")

## Workstream B — E16.B1–B3 representation transfer validation

Representations eligible by the **M15 gate** (GO; GO_DESCRIPTIVE only with `allow_descriptive`) are read from the M15 run. Five-fold held-out retrieval in three pools (all / same position / stopwatch twins), against F0 and random; pre-declared primary = each representation vs F0, same-position MRR (Holm); secondary = all-pool top-1 minus the per-query position-only expectation; CEM context sensitivity; leakage audit. With no eligible representation the step is BLOCKED_MISSING_INPUT and G16-D stays NOT_EVALUATED.

In [ ]:
# E16.B1 / B2 — confirmatory retrieval of the G15-eligible representations + CEM context sensitivity
def step_b1(ctx):
    return m16pipe.run_retrieval(cfg, EXP, DATA16, FEAT, run, ctx)

RET = S.run_step(run, run.ledger, 'E16.B1', step_b1, note='5-fold retrieval: pools, F0, position-only, CEM', requires=['E16.open'])
if RET is not None:
    print(RET['note'])
    display(RET['summary'])
    display(Markdown('**Pre-declared tests (Holm over representations vs F0, same-position MRR)**')); display(RET['tests'])
    if RET['cem'] is not None:
        display(Markdown('**CEM sensitivity**')); display(RET['cem'])

In [ ]:
if RET is not None and RET['summary'].height:
    for pool in ('same_position', 'all', 'twins'):
        if RET['summary'].filter(pl.col('pool') == pool).height:
            fig, dat = m15plots.retrieval_bars(RET['summary'], None, pool)
            run.figure(fig, f'M16_B1_01_retrieval_{pool}', data=dat, question='Does cross-domain similarity survive five folds and position control?',
                       caption=f'Five-fold held-out retrieval, pool {pool}; ticks = uniform chance.')

In [ ]:
# E16.B3 — leakage audit of the eligible representations (only combine_only_inductive ones may become M18 predictors)
def step_b3(ctx):
    if RET is None or not RET['reps']:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'no eligible representation to audit', missing=['E16.B1'])
    return m16pipe.run_leakage(cfg, EXP, RET['reps'], run, ctx)

LEAK = S.run_step(run, run.ledger, 'E16.B3', step_b3, note='leakage audit', requires=['E16.B1'])
if LEAK is not None:
    display(LEAK['verdict'])
    display(LEAK['audit'].filter(pl.col('status') != S.PASS))

In [ ]:
# G16 — gate A–D from the saved tables
def step_g16(ctx):
    return m16pipe.run_gate(run, ctx)

G16 = S.run_step(run, run.ledger, 'G16', step_g16, note='G16 A–D')
if G16 is not None:
    display(G16)
    fig, dat = m15plots.gate_matrix(G16.with_columns(pl.lit('M16').alias('family')), 'G16')
    run.figure(fig, 'M16_G_01_gate_matrix', data=dat, question='Are targets, baseline and cross-domain similarity verified?', caption='G16 A–D statuses.')

## F. Results — gates and pivot

Readings follow the gates document: G16-B FAIL → stop (fix the data contract) · G16-A all task targets FAIL → T0 only · G16-C PRECISE_NULL / INCONCLUSIVE → M18 still runs, the null is the expected reading, the MDE table says what could have been detected · G16-D FAIL → cross-domain similarity claims are dropped (Combine-only representations may still be predictors in M18 if the leakage audit passed).

In [ ]:
if run.has('g16b_gate', 'synthesis'):
    display(run.frame('g16b_gate', 'synthesis'))
else:
    print('G16-B: NOT_EVALUATED (the verification did not produce its gate table this session)')
rep = run.root / 'reports' / 'markdown' / 'm1_audit_report.md'
if rep.is_file():
    display(Markdown(rep.read_text()))

## G. Persistence and hand-off

In [ ]:
run.write_status()
run.save_markdown(run.handoff_markdown(), 'm16_handoff')
for rel in ('manifests/m16_status.json', 'manifests/m1_contract.json', 'manifests/m1_verification.json', 'manifests/target_freeze_m16.json',
            'features/targets', 'predictions/oof_long.parquet', 'tables/target', 'tables/baselines', 'tables/registry', 'tables/detectability',
            'tables/retrieval', 'tables/synthesis', 'diagnostics/leakage_audit.parquet', 'figures/png', 'reports/markdown'):
    p = run.root / rel
    print(('✓ ' if p.exists() else '· ') + str(p))
print('\nSEND BACK: manifests/stages.json, manifests/m16_status.json, manifests/m1_verification.json, manifests/target_freeze_m16.json, logs/*_traceback.txt,')
print('           tables/baselines/m1_reproducibility.csv, tables/target/reliability.csv, tables/target/gate_a.csv, tables/baselines/paired.csv,')
print('           tables/baselines/status.csv, tables/detectability/mde.csv, tables/retrieval/m16_tests.csv, tables/retrieval/leakage_verdict.csv,')
print('           tables/synthesis/m16_gate.csv')